# Learn ImageNet class weights with REPLAY

A minimal classification experiment on `benjamin-paine/imagenet-1k-128x128`:

1. build a small balanced ImageNet-1k training pool,
2. initialize a ViT-Tiny classifier defined in this notebook,
3. train it along a fixed inner trajectory, and
4. learn class weights from a held-out target-class objective with exact REPLAY metagradients.


In [1]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel
from torchvision.transforms import v2

from functional_train import SmoothAdamWConfig, initialize_train_state
from metagrad import InnerBatch, ObjectiveBatch
from replay import replay_metagradient


@dataclass(frozen=True)
class Config:
    seed: int = 0
    dataset: str = "benjamin-paine/imagenet-1k-128x128"
    target_class: int = 0
    train_per_class: int = 2
    objective_examples: int = 16
    image_size: int = 128
    patch_size: int = 16
    batch_size: int = 32
    inner_steps: int = 4
    meta_steps: int = 3
    inner_lr: float = 3e-4
    meta_lr: float = 0.25
    temperature: float = 0.5
    branching_factor: int = 3


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed)
torch.use_deterministic_algorithms(True)
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False

print(device)


c:\Users\luequ\micromamba\envs\torch311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


## Data


In [3]:
dataset = load_dataset(cfg.dataset, keep_in_memory=False)
class_names = dataset["train"].features["label"].names
num_classes = len(class_names)

transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((cfg.image_size, cfg.image_size), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
])


def first_indices(split, counts):
    found = {label: [] for label in counts}
    remaining = sum(counts.values())
    for index, row in enumerate(dataset[split].select_columns(["label"])):
        label = int(row["label"])
        if label in found and len(found[label]) < counts[label]:
            found[label].append(index)
            remaining -= 1
            if remaining == 0:
                return found
    raise ValueError(f"{split} is missing {remaining} requested examples")


def decode(split, indices):
    return torch.stack([
        transform(dataset[split][index]["image"].convert("RGB"))
        for index in indices
    ])


train_rows = first_indices("train", {label: cfg.train_per_class for label in range(num_classes)})
training_images = torch.cat([decode("train", train_rows[label]) for label in range(num_classes)]).to(device)
training_labels = torch.arange(num_classes, device=device).repeat_interleave(cfg.train_per_class)

objective_rows = first_indices("validation", {cfg.target_class: cfg.objective_examples})
objective_batch = ObjectiveBatch(
    decode("validation", objective_rows[cfg.target_class]).to(device),
    torch.full((cfg.objective_examples,), cfg.target_class, dtype=torch.long, device=device),
)

# The groups being weighted are the ImageNet classes themselves.
base_group_masses = torch.full((num_classes,), 1 / num_classes, device=device)
print(f"pool={len(training_labels):,}  objective={len(objective_batch.labels)}  target={class_names[cfg.target_class]}")


ValueError: train is missing 2 requested examples

## ViT-Tiny and fixed inner trajectory


In [ ]:
def patchify(images, patch_size):
    batch, channels, height, width = images.shape
    grid = height // patch_size
    return (
        images.reshape(batch, channels, grid, patch_size, grid, patch_size)
        .permute(0, 2, 4, 3, 5, 1)
        .reshape(batch, grid * grid, channels * patch_size**2)
    )


class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.projection = nn.Linear(dim, dim)

    def forward(self, x):
        batch, tokens, dim = x.shape
        qkv = self.qkv(x).reshape(batch, tokens, 3, self.heads, self.head_dim)
        query, key, value = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        with sdpa_kernel(backends=[SDPBackend.MATH]):
            x = F.scaled_dot_product_attention(query, key, value)
        return self.projection(x.transpose(1, 2).reshape(batch, tokens, dim))


class Block(nn.Module):
    def __init__(self, dim=192, heads=3, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class ViTTiny(nn.Module):
    # ViT-Tiny: 192-dimensional tokens, 12 blocks, 3 attention heads.
    def __init__(self, image_size=128, patch_size=16, num_classes=1000):
        super().__init__()
        dim = 192
        num_patches = (image_size // patch_size) ** 2
        self.patch_size = patch_size
        self.patch_embedding = nn.Linear(3 * patch_size**2, dim)
        self.class_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.position_embedding = nn.Parameter(torch.zeros(1, num_patches + 1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads=3) for _ in range(12)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)
        nn.init.trunc_normal_(self.class_token, std=0.02)
        nn.init.trunc_normal_(self.position_embedding, std=0.02)

    def forward(self, images):
        patches = self.patch_embedding(patchify(images, self.patch_size))
        class_token = self.class_token.expand(images.shape[0], -1, -1)
        x = torch.cat((class_token, patches), dim=1) + self.position_embedding
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x)[:, 0])


model = ViTTiny(cfg.image_size, cfg.patch_size, num_classes).to(device)
initial_state = initialize_train_state(model)
optimizer_config = SmoothAdamWConfig(learning_rate=cfg.inner_lr, eps=1e-4)

generator = torch.Generator().manual_seed(cfg.seed)
order = torch.randperm(len(training_labels), generator=generator)
trajectory = []
for step in range(cfg.inner_steps):
    start = (step * cfg.batch_size) % len(order)
    index = torch.cat((order[start:], order[:start]))[:cfg.batch_size].to(device)
    labels = training_labels[index]
    trajectory.append(InnerBatch(training_images[index], labels, labels))

print(f"parameters={sum(p.numel() for p in model.parameters()):,}  inner_steps={len(trajectory)}")


## Learn class weights through REPLAY


In [ ]:
theta = torch.zeros(num_classes, device=device, requires_grad=True)
weight_history = [torch.softmax(theta.detach() / cfg.temperature, dim=0).cpu()]
objective_history = []

for meta_step in range(cfg.meta_steps):
    objective, gradient = replay_metagradient(
        model,
        initial_state,
        trajectory,
        objective_batch,
        theta,
        base_group_masses,
        optimizer_config,
        cfg.temperature,
        branching_factor=cfg.branching_factor,
    )
    objective_history.append(objective.item())
    with torch.no_grad():
        theta -= cfg.meta_lr * gradient.sign()
    weight_history.append(torch.softmax(theta.detach() / cfg.temperature, dim=0).cpu())
    print(f"{meta_step + 1:>2}: objective={objective.item():.4f}")

weight_history = torch.stack(weight_history)
learned_weights = weight_history[-1]


## Result


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(1, cfg.meta_steps + 1), objective_history, marker="o")
axes[0].set(xlabel="meta-step", ylabel="held-out cross-entropy", title=f"Objective: {class_names[cfg.target_class]}")
axes[0].grid(alpha=0.25)

top = learned_weights.argsort(descending=True)[:20]
axes[1].barh(range(len(top)), learned_weights[top].flip(0))
axes[1].set_yticks(range(len(top)), [class_names[label].split(",")[0] for label in top.flip(0)])
axes[1].set(xlabel="learned class weight", title="Top 20 ImageNet classes")
axes[1].grid(alpha=0.25, axis="x")

fig.tight_layout()
plt.show()

for label in top:
    print(f"{class_names[label]:>30}: {learned_weights[label]:.6f}")


## Standard one-epoch baseline


In [ ]:
from torch.utils.data import DataLoader, Dataset


class ImageNetTrainDataset(Dataset):
    def __init__(self, split, transform):
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.split)

    def __getitem__(self, index):
        example = self.split[index]
        return self.transform(example["image"].convert("RGB")), example["label"]


train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop((cfg.image_size, cfg.image_size), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
train_loader = DataLoader(
    ImageNetTrainDataset(dataset["train"], train_transform),
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

torch.manual_seed(cfg.seed)
baseline_model = ViTTiny(cfg.image_size, cfg.patch_size, num_classes).to(device)
baseline_optimizer = torch.optim.AdamW(baseline_model.parameters(), lr=3e-4, weight_decay=0.05)

baseline_model.train()
running_loss = 0.0
running_steps = 0
for step, (images, labels) in enumerate(train_loader, start=1):
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)

    baseline_optimizer.zero_grad(set_to_none=True)
    loss = F.cross_entropy(baseline_model(images), labels)
    loss.backward()
    baseline_optimizer.step()

    running_loss += loss.item()
    running_steps += 1
    if step % 100 == 0 or step == len(train_loader):
        print(f"epoch 1  step {step}/{len(train_loader)}  loss {running_loss / running_steps:.4f}")
        running_loss = 0.0
        running_steps = 0
